# R2-09 · Despliegue y monitoreo — Profundización (opcional) 🔬

**Formación Pública — Rama R2 · Científico de Datos · Notebook de profundización**

Opcional. Dos preguntas que todo equipo se hace cuando el modelo ya está andando:

1. **¿Dónde se degrada y por qué?** — monitoreo **por estación** (spoiler: una estación «se
   degradó» porque su prueba nunca vio el invierno).
2. **¿Cuándo alertar?** — convertir el pronóstico en **alerta**, eligiendo el umbral con 2025 y
   evaluándolo honestamente en 2026.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
# Reconstruye el modelo y sus predicciones (misma receta que la lección; tarda unos segundos)
import os, urllib.request
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, recall_score, precision_score

for archivo in ["aire_diario.csv", "aire_nuevos.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)
historia = pd.read_csv("aire_diario.csv", parse_dates=["fecha"])
nuevos = pd.read_csv("aire_nuevos.csv", parse_dates=["fecha"])

FEATURES = ["estacion", "pm25_hoy", "pm25_ayer", "media_7d", "mes"]
OBJETIVO = "pm25_manana"

def construir_features(df):
    partes = []
    for estacion, d in df.groupby("estacion"):
        d = d.set_index("fecha").sort_index().asfreq("D")
        d["estacion"] = estacion
        d["pm25_hoy"] = d["pm25"]
        d["pm25_ayer"] = d["pm25"].shift(1)
        d["media_7d"] = d["pm25"].rolling(7, min_periods=5).mean()
        d["mes"] = d.index.month
        d["pm25_manana"] = d["pm25"].shift(-1)
        partes.append(d.reset_index())
    return pd.concat(partes, ignore_index=True)

todo = construir_features(pd.concat([historia, nuevos], ignore_index=True))
completo = todo[FEATURES + [OBJETIVO]].notna().all(axis=1)
entrenamiento = todo[(todo["fecha"] < "2025-01-01") & completo]
prueba = todo[(todo["fecha"].dt.year == 2025) & completo].copy()
prod = todo[(todo["fecha"] >= "2026-01-01") & completo].copy()   # 2026 con predicción y dato real

pre = ColumnTransformer([("est", OneHotEncoder(handle_unknown="ignore"), ["estacion"])], remainder="passthrough")
modelo = Pipeline([("pre", pre), ("bosque", RandomForestRegressor(n_estimators=100, min_samples_leaf=10,
                                                                  random_state=42, n_jobs=-1))])
modelo.fit(entrenamiento[FEATURES], entrenamiento[OBJETIVO])
prueba["pred"] = modelo.predict(prueba[FEATURES])
prod["pred"] = modelo.predict(prod[FEATURES])
print("prueba 2025:", len(prueba), "filas · producción 2026:", len(prod), "filas")

## 1. Monitoreo por estación: ¿dónde se degrada?

El MAE global esconde diferencias. Calcula el MAE **por estación** en la prueba 2025 y en 2026, y la razón `2026 / prueba`. Una razón cercana a 1 = estable; muy sobre 1 = se degradó.

### ✍️ Ejercicio 1 — Tabla de MAE por estación y la peor

In [ ]:
def mae_por_estacion(d):
    return (d["pred"] - d[OBJETIVO]).abs().groupby(d["estacion"]).mean()

por_estacion = pd.DataFrame({"mae_prueba": mae_por_estacion(prueba), "mae_2026": mae_por_estacion(prod)})
por_estacion["razon"] = por_estacion["mae_2026"] / por_estacion["mae_prueba"]
peor_estacion = por_estacion["razon"].idxmax()
print(por_estacion.round(2).sort_values("razon", ascending=False).head(5))
print("Peor estación:", peor_estacion)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    _m = (prod["pred"] - prod[OBJETIVO]).abs().groupby(prod["estacion"]).mean()
    assert np.allclose(por_estacion["mae_2026"].astype(float).values, _m.reindex(por_estacion.index).values)
    assert peor_estacion == por_estacion["razon"].astype(float).idxmax()
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", '(d["pred"] - d[OBJETIVO]).abs().groupby(d["estacion"]).mean(); la peor con .idxmax().')
    print("   Detalle:", e)

## 2. ¿Por qué? Revisa qué midió la prueba

Antes de culpar al modelo, mira **qué contenía la prueba** para esa estación. El PM2.5 de Santiago se dispara en invierno (mayo–agosto). Cuenta, por estación, cuántos días de **invierno** tuvo la prueba 2025. Si una estación no tiene invierno en la prueba, su MAE de prueba solo midió los meses fáciles.

### ✍️ Ejercicio 2 — Días de invierno evaluados en la prueba

In [ ]:
invierno = prueba["fecha"].dt.month.between(5, 8)
dias_invierno_prueba = invierno.groupby(prueba["estacion"]).sum()
sin_invierno = sorted(dias_invierno_prueba[dias_invierno_prueba == 0].index)
print(dias_invierno_prueba.sort_values().head(4))
print("Estaciones SIN invierno en la prueba 2025:", sin_invierno)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    _d = prueba["fecha"].dt.month.between(5, 8).groupby(prueba["estacion"]).sum()
    assert list(sin_invierno) == sorted(_d[_d == 0].index)
    assert peor_estacion in sin_invierno, "¿coincide con la peor estación del ejercicio 1?"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'invierno.groupby(prueba["estacion"]).sum(); luego sorted(dias_invierno_prueba[dias_invierno_prueba == 0].index).')
    print("   Detalle:", e)

## 3. De pronóstico a alerta: el umbral ingenuo

Un uso práctico: **alertar** cuando el pronóstico supera la norma diaria de PM2.5 (50 µg/m³, D.S. 12/2011 del MMA). Mide en la **prueba 2025** qué fracción de los días que realmente superaron la norma fue alertada (*recall*) y qué fracción de las alertas fue correcta (*precision*).

### ✍️ Ejercicio 3 — Recall y precision del umbral 50

In [ ]:
real_prueba = prueba[OBJETIVO] > 50
alerta_50 = prueba["pred"] >= 50
recall_50 = recall_score(real_prueba, alerta_50)
precision_50 = precision_score(real_prueba, alerta_50)
print(f"Días sobre la norma en 2025: {real_prueba.sum()} · recall {recall_50:.2f} · precision {precision_50:.2f}")

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    _a = prueba["pred"] >= 50
    assert abs(recall_50 - recall_score(prueba[OBJETIVO] > 50, _a)) < 1e-9
    assert abs(precision_50 - precision_score(prueba[OBJETIVO] > 50, _a)) < 1e-9
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'alerta_50 = prueba["pred"] >= 50; recall_score(real_prueba, alerta_50); precision_score(real_prueba, alerta_50).')
    print("   Detalle:", e)

## 4. Elegir el umbral con 2025, evaluarlo con 2026

Un umbral de 50 deja pasar muchos días sobre la norma: el modelo tiende a **suavizar los picos**. Para la salud, perder un episodio es peor que una falsa alarma. Elige el umbral **más alto** (entre 20 y 50) que logre **recall ≥ 0,80 en 2025** — y **recién después** mira cómo se comporta en 2026. Elegir el umbral mirando 2026 sería hacer trampa.

### ✍️ Ejercicio 4 — Umbral de alerta validado

In [ ]:
candidatos = range(20, 51)
recall_por_umbral = {u: recall_score(real_prueba, prueba["pred"] >= u) for u in candidatos}
umbral = max(u for u, r in recall_por_umbral.items() if r >= 0.80)
real_2026 = prod[OBJETIVO] > 50
alerta_2026 = prod["pred"] >= umbral
recall_2026 = recall_score(real_2026, alerta_2026)
precision_2026 = precision_score(real_2026, alerta_2026)
print(f"Umbral elegido en 2025: {umbral} µg/m³")
print(f"En 2026: {real_2026.sum()} días sobre la norma · recall {recall_2026:.2f} · precision {precision_2026:.2f} · alertas {alerta_2026.sum()}")

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    _r = {u: recall_score(prueba[OBJETIVO] > 50, prueba["pred"] >= u) for u in range(20, 51)}
    _u = max(u for u, r in _r.items() if r >= 0.80)
    assert umbral == _u, f"el umbral elegido con 2025 debería ser {_u}"
    assert abs(recall_2026 - recall_score(prod[OBJETIVO] > 50, prod["pred"] >= _u)) < 1e-9
    assert abs(precision_2026 - precision_score(prod[OBJETIVO] > 50, prod["pred"] >= _u)) < 1e-9
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", '{u: recall_score(real_prueba, prueba["pred"] >= u) for u in candidatos}; luego max(u for u, r in ... if r >= 0.80).')
    print("   Detalle:", e)

## Para llevar

- Una estación que «se degrada» puede ser una **prueba que no representaba** su realidad: El
  Bosque no reportó entre abril de 2025 y abril de 2026, así que su MAE de prueba solo midió
  verano. **Monitorear también es auditar la prueba.**
- Un umbral de alerta es una **decisión de política**, no un detalle técnico: cambia cuántos
  episodios se pierden y cuántas falsas alarmas se aceptan. Se elige con datos de validación y
  se **declara** en la ficha.

---

*Fin de la profundización de R2-09.*